# V0.1 Execution Spine Lab

问题：一个 LLM ToolCall 进入 Kernel 后发生什么？

这个 notebook 不会一次性跑完整实验。你会逐格看到：model-visible request、observable model response、Kernel tool boundary、Session event log，以及下一次模型可见输入。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v01", mode=MODE)


## Step 1: Setup

创建 Agent、Session 和 `calculator.divide` Tool。注意：Tool 是否对模型可见由 Kernel 的 ToolRegistry 决定。

In [ ]:
_ = lab.setup()

## Step 2: Inspect model-visible request

在运行模型前，先看模型到底能看到什么：system prompt、user message、tool schema。

In [ ]:
_ = lab.show_model_request()

预测一下：模型是直接回答 `4`，还是提出 `calculator.divide` ToolCall？

In [ ]:
_ = lab.model_step()

## Step 3: Kernel executes, not the model

模型只是提出 ToolCall。下一格才是真正的 Kernel boundary：授权、执行、结果归一化、写入 Session。

In [ ]:
_ = lab.kernel_execute_tool()

## Step 4: Next model-visible request

ToolResult 不是模型私自获得的能力，而是 Kernel 生成的下一次可见输入。

In [ ]:
_ = lab.show_next_visible_request()

## Summary

In [ ]:
_ = lab.summary()